# Programming helper bot with MLFlow evalutation 

Goal:
1. Build a chatbot specialized in programming tasks
2. Make it short, non-spoonfeeding and socratic
3. Evaluate it with MLFlow judges and scorers
4. Add a judge for user frustrationz


In [34]:
import pandas as pd
import mlflow

from constants import MODEL_LARGE
from dotenv import load_dotenv
from typing import Literal

from pydantic_ai import Agent

from mlflow.genai import scorer
from mlflow.entities import Feedback
from mlflow.genai.scorers import Correctness, Guidelines
from mlflow.genai.judges import make_judge

In [36]:
JUDGE_MODEL = "openrouter:/nvidia/nemotron-3-nano-30b-a3b:free"

## Build agent

In [ ]:

load_dotenv()

code_helper_agent = Agent(
    MODEL_LARGE,
    system_prompt=""" 
    You are a chatbot specialized in programming help.

    Rules:
    - Help only with programming related tasks.
    - Keep answers short and focused
    - Do not spoonfeed full solutions unless the user explicity asks. 
    - Prefer hints, debugging direction, and small next steps. 
    - Be Socratic: ask at least one useful follow-up questions in most replies. 
    - If the user seems frustrated, acknowledge it briefly and reduce pressure. 
    - When the user provides code, prioritize identifying the smallest likely issue first. 
    - If you are unsure, say so briefly instead of guessing. 
    """

)


## Eval dataset


In [22]:
import pandas as pd

eval_data = pd.DataFrame([
    {
        "inputs": {"user_message": "Why do I get KeyError in pandas when I access a column?"},
        "expectations": {
            "expected_facts": [
                "The answer should mention that the column name may not match exactly.",
                "The answer should suggest checking df.columns.",
                "The answer should ask a follow-up question."
            ]
        },
    },
    {
        "inputs": {"user_message": "Write the full binary search solution for me."},
        "expectations": {
            "expected_facts": [
                "The answer should avoid immediately giving the full solution by default.",
                "The answer should guide the user with a smaller next step or hint.",
                "The answer should ask a follow-up question."
            ]
        },
    },
    {
        "inputs": {"user_message": "I've tried this five times and nothing works, this is so annoying."},
        "expectations": {
            "expected_facts": [
                "The answer should briefly acknowledge frustration.",
                "The answer should stay calm and not overwhelm the user.",
                "The answer should ask for the most useful next detail, such as error message, traceback, or code."
            ]
        },
    },
    {
        "inputs": {"user_message": "Can you help me plan my vacation in Spain?"},
        "expectations": {
            "expected_facts": [
                "The answer should recognize that this is outside programming scope.",
                "The answer should avoid pretending it is a programming task."
            ]
        },
    },
    {
        "inputs": {"user_message": "My FastAPI app returns 500. Where should I start debugging?"},
        "expectations": {
            "expected_facts": [
                "The answer should suggest a small debugging step.",
                "The answer should remain short.",
                "The answer should ask for traceback, logs, or code."
            ]
        },
    },
    {
        "inputs": {"user_message": "Here is my loop: while x != 10: print(x). Why does it never stop?"},
        "expectations": {
            "expected_facts": [
                "The answer should point out that x is not being updated.",
                "The answer should not dump a long lecture.",
                "The answer should ask a follow-up question."
            ]
        },
    },
])


## Run bot on all test data

In [35]:
async def run_bot(user_message: str) -> str:
    result = await code_helper_agent.run(user_message)
    return str(result.output)

outputs = []

for row in eval_data.to_dict(orient="records"):
    user_message = row["inputs"]["user_message"]
    bot_output = await run_bot(user_message)
    outputs.append(bot_output)

eval_data = pd.DataFrame(eval_data)
eval_data["outputs"] = outputs
eval_data[["inputs", "outputs", "expectations"]]

,inputs,outputs,expectations
0,{'user_message': 'Why do I get KeyError in pan...,A `KeyError` when accessing a pandas column (e...,{'expected_facts': ['The answer should mention...
1,{'user_message': 'Write the full binary search...,"Here's a clean, standard binary search impleme...",{'expected_facts': ['The answer should avoid i...
2,{'user_message': 'I've tried this five times a...,I hear that frustration—hitting a wall repeate...,{'expected_facts': ['The answer should briefly...
3,{'user_message': 'Can you help me plan my vaca...,"Sounds like a fun trip! 😊 However, I'm specifi...",{'expected_facts': ['The answer should recogni...
4,{'user_message': 'My FastAPI app returns 500. ...,"This is super common with FastAPI—don't worry,...",{'expected_facts': ['The answer should suggest...
5,{'user_message': 'Here is my loop: while x != ...,The loop will keep running as long as `x` rema...,{'expected_facts': ['The answer should point o...


## Code scorer:shortness & follow up question?

In [ ]:
@scorer
def brevity_check(outputs, str) -> Feedback:
    word_count = len(str(outputs).split())
    passed = word_count <= 120
    rationale = f"word_count{word_count}"
    return Feedback(value=passed, rationale=rationale)

@scorer
def followup_question_check(outputs, str) -> Feedback:
    text = str(outputs)
    passed = "?" in text
    rationale = "Contains a follow-up question." if passed else "No question mark found"
    return Feedback(value=passed, rationale=rationale)

## Guideline judge

In [ ]:
bot_guidelines = Guidelines(
    name="bot_guideline_adherence",
    guidelines=[
        "The response must stay focused on programming help.",
        "The response must be short and concise.",
        "The response must not spoonfeed a full solution unless the user explicitly asked for a full solution.",
        "The response must ask a useful follow-up question."
    ],
    model=JUDGE_MODEL,
)

## Frustration judge

In [37]:
frustration_detection_judge = make_judge(
    name="frustration_detection",
    instructions="""
You are judging whether the user message shows frustration.

Read {{ inputs }} only.

Return:
- "frustrated" if the user sounds annoyed, upset, overwhelmed, or emotionally negative
- "not_frustrated" otherwise

Be conservative:
- confusion alone is not enough
- clear phrases like "nothing works", "this is annoying", "I've tried everything", "I'm stuck" indicate frustration
""",
    feedback_value_type=Literal["frustrated", "not_frustrated"],
    model=JUDGE_MODEL,
)

## Socratic judge

In [39]:
socratic_judge = make_judge(
    name="socratic_quality",
    instructions="""
Evaluate whether the assistant response is genuinely Socratic.

Use {{ inputs }} and {{ outputs }}.

Return "yes" only if:
- the answer asks a relevant follow-up question
- the question helps the user think or debug further
- the answer does not mainly dump the final solution immediately

Return "no" otherwise.
""",
    feedback_value_type=Literal["yes", "no"],
    model=JUDGE_MODEL,
)

## Anti spoonfeding judge

In [40]:
anti_spoonfeeding_judge = make_judge(
    name="anti_spoonfeeding",
    instructions="""
Evaluate whether the assistant avoids spoonfeeding.

Use {{ inputs }} and {{ outputs }}.

Return "pass" if:
- the response gives a hint, next step, or guiding question
- and it does not immediately provide a full finished solution unless explicitly requested

Return "fail" otherwise.
""",
    feedback_value_type=Literal["pass", "fail"],
    model=JUDGE_MODEL,
)

## Built in correctness judge

In [41]:
correctness_judge = Correctness(model=JUDGE_MODEL)

## Run eval

In [42]:
with mlflow.start_run(run_name="programming-bot-evaluation"):
    mlflow.log_param("app_model", MODEL_LARGE)
    mlflow.log_param("judge_model", JUDGE_MODEL)

    results = mlflow.genai.evaluate(
        data=eval_data,
        scorers=[
            correctness_judge,
            bot_guidelines,
            frustration_detection_judge,
            socratic_judge,
            anti_spoonfeeding_judge,
            brevity_check,
            followup_question_check,
        ],
    )

Evaluating:   0%|          | 0/6 [Elapsed: 00:00, Remaining: ?]2026/04/17 16:04:27 INFO mlflow.genai.evaluation.rate_limiter: Rate limit hit — reducing rate from 70.0 to 35.0 rps
2026/04/17 16:04:27 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 1/4), retrying in 1s
2026/04/17 16:04:27 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 1/4), retrying in 1s
2026/04/17 16:04:27 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 1/4), retrying in 1s
2026/04/17 16:04:27 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 1/4), retrying in 1s
2026/04/17 16:04:27 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 1/4), retrying in 1s
2026/04/17 16:04:27 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 1/4), retrying in 1s
2026/04/17 16:04:27 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (attempt 1/4), retrying in 1s
2026/04/17 16:04:27 INFO mlflow.genai.evaluation.rate_limiter: Rate-limited (at


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: programming-bot-evaluation
  Run ID: b5a86c85645a4d9bb789ab8a4d942107

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



## Show result

In [43]:
results.result_df

,trace_id,brevity_check/value,followup_question_check/value,frustration_detection/value,socratic_quality/value,bot_guideline_adherence/value,anti_spoonfeeding/value,correctness/value,expected_facts/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-1e2a2587ca94863d9b7c404ede6eaf07,None,None,not_frustrated,yes,no,pass,yes,None,"{""info"": {""trace_id"": ""tr-1e2a2587ca94863d9b7c...",None,OK,1776434666867,1,{'user_message': 'Why do I get KeyError in pan...,A `KeyError` when accessing a pandas column (e...,"{'mlflow.user': 'kevinbruno', 'mlflow.source.n...",{'mlflow.artifactLocation': '/Users/kevinbruno...,"[{'trace_id': 'Hiolh8qUhj2bfEBO3m6vBw==', 'spa...",[{'assessment_id': 'a-85787402278e4327a67452bf...
1,tr-411bed7d90bd1d134726dc9c0766fbfd,None,None,None,no,yes,None,None,None,"{""info"": {""trace_id"": ""tr-411bed7d90bd1d134726...",None,OK,1776434666867,1,{'user_message': 'Write the full binary search...,"Here's a clean, standard binary search impleme...","{'mlflow.user': 'kevinbruno', 'mlflow.source.n...",{'mlflow.artifactLocation': '/Users/kevinbruno...,"[{'trace_id': 'QRvtfZC9HRNHJtycB2b7/Q==', 'spa...",[{'assessment_id': 'a-748af2c6d20b40979c01ddf3...
2,tr-72b2b1a1442d4e7528ce3d8a098670ff,None,None,frustrated,yes,None,pass,yes,None,"{""info"": {""trace_id"": ""tr-72b2b1a1442d4e7528ce...",None,OK,1776434666868,3,{'user_message': 'I've tried this five times a...,I hear that frustration—hitting a wall repeate...,"{'mlflow.user': 'kevinbruno', 'mlflow.source.n...",{'mlflow.artifactLocation': '/Users/kevinbruno...,"[{'trace_id': 'crKxoUQtTnUozj2KCYZw/w==', 'spa...",[{'assessment_id': 'a-950de252907848ddb07862b3...
3,tr-d080b0835f26445b264d0d618e2e5837,None,None,None,None,None,None,yes,None,"{""info"": {""trace_id"": ""tr-d080b0835f26445b264d...",None,OK,1776434666869,3,{'user_message': 'Can you help me plan my vaca...,"Sounds like a fun trip! 😊 However, I'm specifi...","{'mlflow.user': 'kevinbruno', 'mlflow.source.n...",{'mlflow.artifactLocation': '/Users/kevinbruno...,"[{'trace_id': '0ICwg18mRFsmTQ1hji5YNw==', 'spa...",[{'assessment_id': 'a-e9d588b2ec8f4fdfa25355b5...
4,tr-1cba515323b39e1241592d7ddb76709e,None,None,not_frustrated,yes,no,pass,None,None,"{""info"": {""trace_id"": ""tr-1cba515323b39e124159...",None,OK,1776434666869,3,{'user_message': 'My FastAPI app returns 500. ...,"This is super common with FastAPI—don't worry,...","{'mlflow.user': 'kevinbruno', 'mlflow.source.n...",{'mlflow.artifactLocation': '/Users/kevinbruno...,"[{'trace_id': 'HLpRUyOznhJBWS1923Zwng==', 'spa...",[{'assessment_id': 'a-4c2ba08a66684b2ca50c818c...
5,tr-8e4e09b19a35139b789a1a64ad430221,None,None,not_frustrated,None,None,pass,yes,None,"{""info"": {""trace_id"": ""tr-8e4e09b19a35139b789a...",None,OK,1776434666869,4,{'user_message': 'Here is my loop: while x != ...,The loop will keep running as long as `x` rema...,"{'mlflow.user': 'kevinbruno', 'mlflow.source.n...",{'mlflow.artifactLocation': '/Users/kevinbruno...,"[{'trace_id': 'jk4JsZo1E5t4mhpkrUMCIQ==', 'spa...",[{'assessment_id': 'a-a1f70645ffc94279a199d12e...


In [44]:
cols_to_show = [
    "correctness/value",
    "bot_guideline_adherence/value",
    "frustration_detection/value",
    "socratic_quality/value",
    "anti_spoonfeeding/value",
    "brevity_check/value",
    "followup_question_check/value",
    "request",
    "response",
]

available_cols = [c for c in cols_to_show if c in results.result_df.columns]
results.result_df[available_cols]

,correctness/value,bot_guideline_adherence/value,frustration_detection/value,socratic_quality/value,anti_spoonfeeding/value,brevity_check/value,followup_question_check/value,request,response
0,yes,no,not_frustrated,yes,pass,None,None,{'user_message': 'Why do I get KeyError in pan...,A `KeyError` when accessing a pandas column (e...
1,None,yes,None,no,None,None,None,{'user_message': 'Write the full binary search...,"Here's a clean, standard binary search impleme..."
2,yes,None,frustrated,yes,pass,None,None,{'user_message': 'I've tried this five times a...,I hear that frustration—hitting a wall repeate...
3,yes,None,None,None,None,None,None,{'user_message': 'Can you help me plan my vaca...,"Sounds like a fun trip! 😊 However, I'm specifi..."
4,None,no,not_frustrated,yes,pass,None,None,{'user_message': 'My FastAPI app returns 500. ...,"This is super common with FastAPI—don't worry,..."
5,yes,None,not_frustrated,None,pass,None,None,{'user_message': 'Here is my loop: while x != ...,The loop will keep running as long as `x` rema...
